# Finalizar, salvar e carregar

O caminho do campeão até a produção: reajuste no histórico permitido, previsão, avaliação única no teste selado, persistência com manifesto e recarga em outro processo.

In [1]:
import autohybts

print("AutoHybTS", autohybts.__version__)

AutoHybTS 0.0.1


## 1. Experimento e campeão

In [2]:
from autohybts import AutoHybridExperiment
from autohybts.datasets import get_data

y = get_data("synthetic_daily")
exp = AutoHybridExperiment(fh=14, session_id=42).fit(y)
best = exp.compare_models().best

## 2. Finalizar

A estrutura e os hiperparâmetros ficam congelados; só os coeficientes são reajustados no histórico anterior ao teste, e os resíduos do corretor são regenerados com a mesma política.

In [3]:
final = exp.finalize_model(best)
print("estrutura:", final.spec)
print("treinado até:", final.training_cutoff)

estrutura: CandidateSpec(family='tree', strategy='standalone', base=ModelSpec(name='random_forest', params_json='{"lags":7}'), corrector=None, window_size=None, alpha=None, residual_source=None, variant='A', members=(), options_json='{}', components=(), candidate_id='1f8d8f124a8728e54358f736b6cef1744bd8df58ebf5b13403a2dd2723163f06')
treinado até: 2025-12-16 00:00:00


## 3. Prever

In [4]:
forecast = exp.predict_model(final)
forecast.predictions

,series_id,origin,horizon,y_pred
timestamp,,,,
2025-12-17,__single__,2025-12-16,1,73.378733
2025-12-18,__single__,2025-12-16,2,71.898770
2025-12-19,__single__,2025-12-16,3,70.122112
2025-12-20,__single__,2025-12-16,4,68.294473
2025-12-21,__single__,2025-12-16,5,68.671440
2025-12-22,__single__,2025-12-16,6,70.818109
2025-12-23,__single__,2025-12-16,7,73.083289
2025-12-24,__single__,2025-12-16,8,73.790339
2025-12-25,__single__,2025-12-16,9,72.286542


## 4. Avaliar no teste selado

Só uma chamada por modelo finalizado. Usar esse número para voltar e mudar a configuração transformaria o teste em validação.

In [5]:
external = exp.evaluate_external(final)
external.metrics

,MASE,MAE,RMSE,sMAPE
model,,,,
random_forest(lags=7),0.704631,1.103425,1.273037,0.015608


## 5. Salvar e carregar

`save_model` grava o modelo com um manifesto (versões, hashes, `training_cutoff`). `load_model` lê o manifesto sem executar nada; `trusted=True` declara confiança no arquivo e libera a desserialização. Nunca use `trusted=True` com artefatos de origem desconhecida.

In [6]:
from pathlib import Path

from autohybts import load_model, save_model

path = save_model(final, Path("artifacts") / "champion")
restored = load_model(path, trusted=True)
restored.manifest

{'model_id': '89f3c27029c3f5f7b85c9555b47a6096c87e2b9ca06ae023b825a277ae476d2b',
 'spec': {'candidate_id': '1f8d8f124a8728e54358f736b6cef1744bd8df58ebf5b13403a2dd2723163f06',
  'label': 'random_forest(lags=7)',
  'describe': {'family': 'tree',
   'strategy': 'standalone',
   'base': {'name': 'random_forest', 'params': {'lags': 7}},
   'corrector': None,
   'window_size': None,
   'alpha': None,
   'residual_source': None,
   'variant': 'A'}},
 'training_cutoff': '2025-12-16 00:00:00',
 'series_id': '__single__',
 'frequency': 'D',
 'n_observations': 716,
 'history_fingerprint': '82900104735217070dc3caa1b8f4585519eb917c4d46a371f691b45e59e54b0a',
 'plan_fingerprint': '31e3290aeac1df36259d888e6839bdaa9aed875ccb42256b8c7683dab7eee0df',
 'seed': 251114467,
 'experiment': {'fh': 14,
  'session_id': 42,
  'fold': 'auto',
  'fold_strategy': 'expanding',
  'window_size': None,
  'seasonal_period': None,
  'test_size': None,
  'profile': 'standard',
  'variant': 'A',
  'budget': 'small',
  'max_

In [7]:
again = exp.predict_model(restored)
(again.predictions["y_pred"] - forecast.predictions["y_pred"]).abs().max()

np.float64(0.0)

## 6. Previsão operacional

Para prever datas que ainda não aconteceram, use `test_size=0`: não há teste selado, e o modelo finalizado é reajustado em todo o histórico.

In [8]:
production = AutoHybridExperiment(fh=14, session_id=42, test_size=0).fit(y)
model = production.finalize_model(production.compare_models().best)
model.predict(fh=14)

,series_id,origin,horizon,y_pred
timestamp,,,,
2025-12-31,__single__,2025-12-30,1,73.104426
2026-01-01,__single__,2025-12-30,2,71.504198
2026-01-02,__single__,2025-12-30,3,69.037189
2026-01-03,__single__,2025-12-30,4,67.581331
2026-01-04,__single__,2025-12-30,5,68.252915
2026-01-05,__single__,2025-12-30,6,70.566708
2026-01-06,__single__,2025-12-30,7,72.801684
2026-01-07,__single__,2025-12-30,8,73.296713
2026-01-08,__single__,2025-12-30,9,71.700737
